## Set up

In [1]:
%pip install neo4j
%pip install matplotlib


[notice] A new release of pip is available: 23.2.1 -> 23.3.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 23.3.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Imports

In [4]:
from neo4j import GraphDatabase

#### Load classes

In [ ]:
%run ../commons/models.py

### Establish connection and create driver

In [5]:
uri = "bolt://0.0.0.0:7687"
username = "neo4j"
password = "111122223333"
driver = GraphDatabase.driver(uri, auth=(username, password))

#### Get the image data

In [3]:
query = """
    MATCH (target {image_id: '7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47'})-[*0..5]-(connectedNode)
    WHERE NOT "Line" IN labels(connectedNode)
    RETURN connectedNode
"""



with driver.session() as session:
    result = session.execute_read(execute_query, query)
    for record in result:
        print(record)

NameError: name 'execute_query' is not defined

#### Clean up

In [7]:
def execute_query(tx, query):
    result = tx.run(query)
    return [record for record in result]

#### Playground

In [38]:
query = """
MATCH (cp: CriticalPoint {reason: 'First point'})--(ap: AnglePoint)
RETURN cp, ap
"""

prio_one_image_id = None;

first_critical_points = {}
with driver.session() as session:
    result = session.execute_read(execute_query, query)
    for record in result:
        if prio_one_image_id is None:
            prio_one_image_id = record['cp']['image_id']
        cp = record['cp']
        image_id = cp['image_id']
        if image_id not in first_critical_points:
            first_critical_points[image_id] = []
        first_critical_points[image_id].append(CriticalPoint(uuid=cp['uuid'], reason=cp['reason'], image_id=image_id))
        
if len(first_critical_points) != 2:
    raise Exception('First critical points not found');

for image_id, critical_points in first_critical_points.items():
    if image_id != prio_one_image_id:
        result = compare_nodes(first_critical_points[prio_one_image_id][0], critical_points[0])
        print(f"Comparison result between {prio_one_image_id} and {image_id}: {result}")

Comparison result between 5368b59c-1bfc-4614-9d7b-f4901d0c8db4 and 4e96eab9-2c9d-4fef-915f-4671589dea55: 1


In [35]:
query = """
MATCH (cp: CriticalPoint {reason: 'First point'})--(ap: AnglePoint)--(apLocation: AnglePointLocation)
RETURN cp, ap, apLocation
"""

first_ap_location = None
other_ap_locations = []
with driver.session() as session:
    result = session.execute_read(execute_query, query)
    for record in result:
        apLocation = record['apLocation']
        print(apLocation)

<Node element_id='4:b6d943ed-a78d-4853-a283-1394caf2f399:26' labels=frozenset({'AnglePointLocation'}) properties={'x': 46, 'y': 14, 'angle': 60}>
<Node element_id='4:b6d943ed-a78d-4853-a283-1394caf2f399:85' labels=frozenset({'AnglePointLocation'}) properties={'x': 47, 'y': 18, 'angle': 75}>


In [32]:
uuid = first_critica_points[0].uuid
print(f'Updating uuid: {uuid}')
query = f"""
MATCH (cp:CriticalPoint {{uuid: '{uuid}'}})
SET cp.weight = coalesce(cp.weight, 0) + {result}
"""

with driver.session() as session:
    session.run(query)

Updating uuid: fbe3daa7-4314-4dc8-a1a9-5e9bd348f346


In [ ]:
uuid = first_critica_points[0].uuid
print(f'Updating uuid: {uuid}')
query = f"""
MATCH (cp:CriticalPoint {{uuid: '{uuid}'}})
SET cp.weight = coalesce(cp.weight, 0) + {result}
"""

with driver.session() as session:
    session.run(query)

In [33]:
# %load ../commons/models.py
from dataclasses import dataclass
from enum import Enum

class Reason(Enum):
    FIRST_POINT = "First point"
    FIRST_LINE = "First Line"

@dataclass(frozen=True)
class CriticalPoint:
    uuid: str
    image_id: str
    reason: Reason
    connectedVerts = {}
    
@dataclass(frozen=True)
class AnglePointLocation:
    angle: int
    x: int
    y: int


In [37]:
def compare_angle_points(ap1: AnglePointLocation, ap2: AnglePointLocation):
    ap_loc_weight = 0
    if ap1.angle == ap2.angle:
        ap_loc_weight += 1
    if ap1.x == ap2.x:
        ap_loc_weight += 1
    if ap1.y == ap2.y:
        ap_loc_weight += 1
    return ap_loc_weight

def compare_critical_points(cp1: CriticalPoint, cp2: CriticalPoint):
    return 1 if (cp1.reason == cp2.reason) else 0

def compare_nodes(node1, node2):
    if isinstance(node1, CriticalPoint):
        return compare_critical_points(node1, node2)
    elif isinstance(node1, AnglePointLocation):
        return compare_angle_points(node1, node2)
    else:
        return "Unsupported node type"


#### Close the driver

In [ ]:
driver.close()